# Clean KTTS VITS v1 listening evaluation
Generates fixed Kashmiri samples from the best 200-epoch VITS checkpoint.


In [ ]:
!pip install -q 'coqui-tts==0.28.0' 'coqui-tts-trainer>=0.4.0,<0.5.0' soundfile
from pathlib import Path
import subprocess
artifact=next(Path('/kaggle/input').rglob('ktts-vits-clean-v1.zip'))
subprocess.run(['unzip','-o',str(artifact),'model.pth','config.json','training-report.json','audit-report.json','-d','/kaggle/working/vits'],check=True)
print('VITS artifact:',artifact)


In [ ]:
from TTS.api import TTS
from pathlib import Path
import torch
texts=['السلام علیکم، تُہۍ کِتھ پٲٹھۍ چھِو؟','مےٚ ناو چھُ سمیر، بیٚہ چھُس کویل پُلوامہ پٮ۪ٹھ۔','اَسہِ چھِ کٲشُر زبان واریاہ پیٛارِ۔','کٲشُر زَبان چھِ اَسہِ واریاہ پیٛارۍ۔']
out=Path('/kaggle/working/evaluation/vits_clean_v1'); out.mkdir(parents=True,exist_ok=True)
tts=TTS(model_path='/kaggle/working/vits/model.pth',config_path='/kaggle/working/vits/config.json',gpu=torch.cuda.is_available())
for index,text in enumerate(texts,1):
    target=out/f'sample_{index:02d}.wav'
    tts.tts_to_file(text=text,file_path=str(target))
    print(target,text)


In [ ]:
import json,numpy as np,soundfile as sf
from pathlib import Path
rows=[]
for wav in sorted(Path('/kaggle/working/evaluation').rglob('*.wav')):
    y,sr=sf.read(wav); rows.append({'file':wav.name,'seconds':round(len(y)/sr,3),'sampleRate':sr,'peak':round(float(np.max(np.abs(y))),5),'rms':round(float(np.sqrt(np.mean(y*y))),5),'clippedSamples':int(np.sum(np.abs(y)>=0.999))})
report={'model':'KTTS Clean Coqui VITS v1','epochs':200,'sentences':texts,'measurements':rows,'productionDeployed':False,'humanReviewRequired':True}
Path('/kaggle/working/evaluation/report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(report,ensure_ascii=False,indent=2))
!cd /kaggle/working && zip -qr ktts-vits-clean-v1-evaluation.zip evaluation
